# XGBoost Experiment — IEEE-CIS Fraud Detection
Full pipeline: Cleaning → Feature Engineering → Feature Selection → Training → MLflow Logging

## 0. Setup & Imports

In [ ]:
# ── STEP 1: Auth BEFORE any dagshub import ──
from kaggle_secrets import UserSecretsClient
import os
os.environ['DAGSHUB_USER_TOKEN'] = UserSecretsClient().get_secret('DAGSHUB_TOKEN')
print('Token loaded!')

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'mlflow', 'dagshub', 'optuna', '--quiet'], capture_output=True)

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import dagshub
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.base import BaseEstimator, TransformerMixin

import xgboost as xgb
print('All imports successful!')
print(f'XGBoost version: {xgb.__version__}')

In [ ]:
dagshub.init(repo_owner='ekali-star', repo_name='fraud-detection-ml', mlflow=True)
mlflow.set_experiment('XGBoost_Training')
print('Connected:', mlflow.get_tracking_uri())

## 1. Data Loading

In [ ]:
# Fixed path — dataset lives under /competitions/ on Kaggle
BASE_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'

train_transaction = pd.read_csv(BASE_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(BASE_PATH + 'train_identity.csv')
test_transaction  = pd.read_csv(BASE_PATH + 'test_transaction.csv')
test_identity     = pd.read_csv(BASE_PATH + 'test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity,  on='TransactionID', how='left')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Fraud rate:  {train.isFraud.mean():.4f}')

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='XGBoost_Cleaning') as cleaning_run:

    missing_pct  = train.isnull().mean().sort_values(ascending=False)
    high_missing = missing_pct[missing_pct > 0.9].index.tolist()
    print(f'Columns with >90% missing: {len(high_missing)}')

    train_clean = train.drop(columns=high_missing)
    test_clean  = test.drop(columns=[c for c in high_missing if c in test.columns])

    constant_cols = [c for c in train_clean.columns
                     if train_clean[c].nunique(dropna=False) <= 1]
    train_clean.drop(columns=constant_cols, inplace=True)
    test_clean.drop(columns=[c for c in constant_cols if c in test_clean.columns], inplace=True)
    print(f'Constant columns removed: {len(constant_cols)}')

    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in train_clean.columns:
            top = train_clean[col].value_counts().nlargest(10).index
            train_clean[col] = train_clean[col].where(train_clean[col].isin(top), 'other')
            if col in test_clean.columns:
                test_clean[col] = test_clean[col].where(test_clean[col].isin(top), 'other')

    mlflow.log_param('high_missing_threshold', 0.9)
    mlflow.log_param('dropped_high_missing_cols', len(high_missing))
    mlflow.log_param('dropped_constant_cols', len(constant_cols))
    mlflow.log_metric('train_rows', train_clean.shape[0])
    mlflow.log_metric('train_cols_after_cleaning', train_clean.shape[1])

    missing_report = missing_pct.reset_index()
    missing_report.columns = ['column', 'missing_pct']
    missing_report.to_csv('missing_report.csv', index=False)
    mlflow.log_artifact('missing_report.csv')

    print(f'Shape after cleaning: {train_clean.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='XGBoost_Feature_Engineering') as fe_run:

    def feature_engineering(df):
        df = df.copy()
        df['hour']         = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']  = (df['TransactionDT'] / (3600 * 24)) % 7
        df['day_of_month'] = (df['TransactionDT'] / (3600 * 24)) % 30
        df['is_night']     = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents']   = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['TransactionAmt_isround'] = (df['TransactionAmt_cents'] == 0).astype(int)
        for grp_col in ['card1', 'card2', 'addr1']:
            if grp_col in df.columns:
                col_mean = df.groupby(grp_col)['TransactionAmt'].transform('mean')
                col_std  = df.groupby(grp_col)['TransactionAmt'].transform('std')
                df[f'{grp_col}_amt_mean'] = col_mean
                df[f'{grp_col}_amt_std']  = col_std
                df[f'{grp_col}_amt_z']    = (df['TransactionAmt'] - col_mean) / (col_std + 1e-5)
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        if 'card4' in df.columns and 'card6' in df.columns:
            df['card_combo'] = df['card4'].astype(str) + '_' + df['card6'].astype(str)
        df['nan_count'] = df.isnull().sum(axis=1)
        return df

    train_fe = feature_engineering(train_clean)
    test_fe  = feature_engineering(test_clean)

    TARGET    = 'isFraud'
    DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]

    cat_cols = [c for c in train_fe.select_dtypes(include='object').columns
                if c not in DROP_COLS]

    label_encoders = {}
    for col in cat_cols:
        le = LabelEncoder()
        # only use values that appear in train for fitting
        le.fit(train_fe[col].fillna('missing').astype(str))
        train_fe[col] = le.transform(train_fe[col].fillna('missing').astype(str))
        if col in test_fe.columns:
            test_fe[col] = test_fe[col].fillna('missing').astype(str).map(
                lambda x: x if x in le.classes_ else le.classes_[0]
            )
            test_fe[col] = le.transform(test_fe[col])
        label_encoders[col] = le

    mlflow.log_param('categorical_cols_encoded', len(cat_cols))
    mlflow.log_metric('total_features_after_fe', train_fe.shape[1])
    print(f'Shape after FE: {train_fe.shape}')
    print(f'Label encoded {len(cat_cols)} categorical columns')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='XGBoost_Feature_Selection') as fs_run:

    feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]
    X = train_fe[feature_cols].fillna(-999)
    y = train_fe[TARGET]

    base_xgb = xgb.XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        tree_method='gpu_hist',
        eval_metric='auc', random_state=42, verbosity=0
    )
    base_xgb.fit(X, y)

    importances = pd.Series(base_xgb.feature_importances_, index=feature_cols)
    importances_sorted = importances.sort_values(ascending=False)

    TOP_N = 150
    selected_features_importance = importances_sorted.head(TOP_N).index.tolist()

    fig, ax = plt.subplots(figsize=(10, 8))
    importances_sorted.head(30).plot(kind='barh', ax=ax)
    ax.set_title('Top 30 Feature Importances (XGBoost)')
    plt.tight_layout()
    plt.savefig('xgb_feature_importance.png', dpi=100)
    mlflow.log_artifact('xgb_feature_importance.png')
    plt.show()

    # Remove highly correlated features
    corr_matrix = X[selected_features_importance].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
    selected_features_final = [f for f in selected_features_importance if f not in high_corr_cols]

    mlflow.log_param('fs_method', 'xgb_importance + correlation_filter')
    mlflow.log_param('top_n_importance', TOP_N)
    mlflow.log_param('high_corr_threshold', 0.95)
    mlflow.log_metric('features_before_fs', len(feature_cols))
    mlflow.log_metric('features_after_importance', len(selected_features_importance))
    mlflow.log_metric('features_removed_corr', len(high_corr_cols))
    mlflow.log_metric('features_final', len(selected_features_final))

    print(f'Features: {len(feature_cols)} → {len(selected_features_final)} selected')

    X_selected = X[selected_features_final]

    # Safe test selection — handle missing columns
    test_features = [f for f in selected_features_final if f in test_fe.columns]
    X_test_selected = test_fe[test_features].fillna(-999)
    for col in selected_features_final:
        if col not in X_test_selected.columns:
            X_test_selected[col] = -999
    X_test_selected = X_test_selected[selected_features_final]

## 5. Training

### 5a. Underfitted Baseline

In [ ]:
with mlflow.start_run(run_name='XGBoost_Underfitted_Baseline'):
    params_underfit = {
        'n_estimators': 50, 'max_depth': 2, 'learning_rate': 0.3,
        'tree_method': 'gpu_hist', 'eval_metric': 'auc',
        'random_state': 42, 'verbosity': 0
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        xgb.XGBClassifier(**params_underfit),
        X_selected, y, cv=cv, scoring='roc_auc'
    )
    mlflow.log_params(params_underfit)
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('cv_auc_std',  cv_scores.std())
    mlflow.log_param('note', 'intentionally_underfitted_shallow_tree')
    print(f'[UNDERFITTED] CV AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print('Analysis: Low depth + few estimators = high bias, underfitting expected')

### 5b. Overfitted Model

In [ ]:
with mlflow.start_run(run_name='XGBoost_Overfitted'):
    params_overfit = {
        'n_estimators': 2000, 'max_depth': 12, 'learning_rate': 0.3,
        'subsample': 1.0, 'colsample_bytree': 1.0, 'min_child_weight': 1,
        'reg_alpha': 0, 'reg_lambda': 0,
        'tree_method': 'gpu_hist', 'eval_metric': 'auc',
        'random_state': 42, 'verbosity': 0
    }
    model_overfit = xgb.XGBClassifier(**params_overfit)
    model_overfit.fit(X_selected, y)
    train_auc = roc_auc_score(y, model_overfit.predict_proba(X_selected)[:, 1])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        xgb.XGBClassifier(**params_overfit),
        X_selected, y, cv=cv, scoring='roc_auc'
    )
    mlflow.log_params(params_overfit)
    mlflow.log_metric('train_auc', train_auc)
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('train_val_gap', train_auc - cv_scores.mean())
    mlflow.log_param('note', 'intentionally_overfitted_deep_tree_no_regularization')
    print(f'[OVERFITTED] Train AUC: {train_auc:.4f} | CV AUC: {cv_scores.mean():.4f}')
    print(f'Gap: {train_auc - cv_scores.mean():.4f} — large gap indicates overfitting')

### 5c. Optuna Hyperparameter Search

In [ ]:
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 300, 1500),
        'max_depth':        trial.suggest_int('max_depth', 4, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 5.0),
        'tree_method':      'gpu_hist',
        'eval_metric':      'auc',
        'random_state':     42,
        'verbosity':        0
    }
    # 3-fold for speed
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        xgb.XGBClassifier(**params),
        X_selected, y, cv=cv, scoring='roc_auc'
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, show_progress_bar=True)

best_params = study.best_params
best_params.update({'tree_method': 'gpu_hist', 'eval_metric': 'auc',
                    'random_state': 42, 'verbosity': 0})
print(f'Best trial AUC: {study.best_value:.4f}')
print(f'Best params: {best_params}')

### 5d. Final CV + Pipeline

In [ ]:
with mlflow.start_run(run_name='XGBoost_Final_CV') as final_run:

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_selected))
    fold_aucs  = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X_selected, y)):
        X_tr, X_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(**best_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            early_stopping_rounds=50,
            verbose=False
        )
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_pred
        test_preds += model.predict_proba(X_test_selected)[:, 1] / 5
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_aucs.append(fold_auc)
        print(f'  Fold {fold+1}: AUC = {fold_auc:.4f}')

    oof_auc = roc_auc_score(y, oof_preds)
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    mlflow.log_metric('cv_auc_std',  np.std(fold_aucs))
    for i, auc in enumerate(fold_aucs):
        mlflow.log_metric(f'fold_{i+1}_auc', auc)
    print(f'OOF AUC: {oof_auc:.4f}')
    print(f'CV  AUC: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')

In [ ]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, selected_features=None):
        self.selected_features = selected_features
        self.label_encoders_   = {}
        self.cat_cols_         = []

    def fit(self, X, y=None):
        df = self._engineer(X.copy())
        self.cat_cols_ = df.select_dtypes(include='object').columns.tolist()
        for col in self.cat_cols_:
            le = LabelEncoder()
            le.fit(df[col].fillna('missing').astype(str))
            self.label_encoders_[col] = le
        return self

    def transform(self, X):
        df = self._engineer(X.copy())
        for col in self.cat_cols_:
            if col in df.columns:
                le = self.label_encoders_[col]
                vals = df[col].fillna('missing').astype(str)
                vals = vals.map(lambda x: x if x in le.classes_ else le.classes_[0])
                df[col] = le.transform(vals)
        if self.selected_features:
            avail = [f for f in self.selected_features if f in df.columns]
            df = df[avail]
        return df.fillna(-999)

    def _engineer(self, df):
        df['hour']         = (df['TransactionDT'] / 3600) % 24
        df['day_of_week']  = (df['TransactionDT'] / (3600 * 24)) % 7
        df['day_of_month'] = (df['TransactionDT'] / (3600 * 24)) % 30
        df['is_night']     = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
        df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_cents']   = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
        df['TransactionAmt_isround'] = (df['TransactionAmt_cents'] == 0).astype(int)
        df['nan_count'] = df.isnull().sum(axis=1)
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['email_match'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
        if 'card4' in df.columns and 'card6' in df.columns:
            df['card_combo'] = df['card4'].astype(str) + '_' + df['card6'].astype(str)
        return df


X_raw_train = train.drop(columns=['isFraud', 'TransactionID', 'TransactionDT'], errors='ignore')
y_train_raw = train['isFraud']

final_xgb_pipeline = Pipeline([
    ('preprocessor', FraudPreprocessor(selected_features=selected_features_final)),
    ('classifier',   xgb.XGBClassifier(**best_params))
])
final_xgb_pipeline.fit(X_raw_train, y_train_raw)
print('Pipeline fitted on full training data!')

In [ ]:
with mlflow.start_run(run_name='XGBoost_Pipeline_Registry'):
    mlflow.log_params(best_params)
    mlflow.log_metric('oof_auc', oof_auc)
    mlflow.log_metric('cv_auc_mean', np.mean(fold_aucs))
    mlflow.sklearn.log_model(
        sk_model=final_xgb_pipeline,
        artifact_path='xgboost_fraud_pipeline',
        registered_model_name='XGBoost_Fraud_Pipeline'
    )
    print('Pipeline registered as: XGBoost_Fraud_Pipeline')

np.save('xgb_test_preds.npy', test_preds)
print('Test predictions saved!')